# Model Overview and Data Generating Process

Assume the latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z}_F + \boldsymbol{\Lambda}_t \mathbf{M}_F,
$$ 
where $t\in 1,\ldots, T$ and the matrices $\boldsymbol{\Theta}_t$ and $\boldsymbol{\Lambda}_t$ are:
$$
\begin{pmatrix}
\theta_{t_{11}}&\dots &\theta_{t_{1p}} \\
\vdots & & \vdots \\
\theta_{t_{m1}} &\dots&\theta_{t_{mp}}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\lambda_{t_{11}}&\dots &\lambda_{t_{1f}} \\
\vdots & & \vdots \\
\lambda_{t_{m1}} &\dots&\lambda_{t_{mf}}
\end{pmatrix}
$$ 
and the matrices $\mathbf{Z}_F$ and $\mathbf{M}_F$ are
$$
\begin{pmatrix}
Z_{11}&\dots & Z_{N,1} \\
\vdots & & \vdots \\
Z_{1p} &\dots& Z_{N, p}
\end{pmatrix} \quad\text{and}\quad
\begin{pmatrix}
\mu_{11}&\dots & \mu_{N,1} \\
\vdots & & \vdots \\
\mu_{1p} &\dots& \mu_{N, p}
\end{pmatrix}
$$ 
In the data generating process, the observed and latent loadings are updated at each time step. This represents changes in feature importance in time. Intially, loading matrices $\mathbf{\Theta}_0$ and $\mathbf{\Lambda}_0$ are specified, each matrix entry drawn from $N(0,1)$. Step sizes $\delta\mathbf{\Theta}$ and $\delta\mathbf{\Lambda}$ are generated in the same way. The outcomes are generated recursively using:
$$
\mathbf{\Theta}_t = \mathbf{\Theta}_{t-1} + c \delta\mathbf{\Theta}\quad\text{and}\quad\mathbf{\Lambda}_t = \mathbf{\Lambda}_{t-1} + c \delta\mathbf{\Lambda},
$$
where at each step, $c$ is the absolute value of a draw from a $N(0,1)$ and is different for $\Theta$ and $\Lambda$.

# Data Generating Code

In [228]:
import numpy as np
import torch
import learnQ as lq

torch.manual_seed(42)
np.random.seed(42)

def sim_Data(T, N, m, p, f, with_covariates):
    Outcomes = []
    Thetas = []
    Lambdas = []
    latent_terms = []

    # Fixed base loadings
    Theta_base = torch.randn(m, p, dtype=torch.float64)
    Lambda_base = torch.randn(m, f, dtype=torch.float64)

    # step directions
    Theta_step = torch.randn(m, p, dtype=torch.float64)
    Lambda_step = torch.randn(m, f, dtype=torch.float64)

    for t in range(T):
        if t == 0:
            # start out at the base, before taking any steps
            Theta_t = Theta_base
            Lambda_t = Lambda_base
        else:
            # take a step, always in the same direction, but by varying amounts
            Theta_t = Theta_base + (Theta_step * torch.abs(torch.randn(1)))
            Lambda_t = Lambda_base + (Lambda_step * torch.abs(torch.randn(1)))
        # each time point append
        Thetas.append(Theta_t)
        Lambdas.append(Lambda_t)

    # if we aren't using covariates, just do the loadings for the latent factors
    Z_F = torch.randn(p, N, dtype=torch.float64) * bool(with_covariates)  # (p x N)
    M_F= torch.randn(f, N, dtype=torch.float64)  # (f x N)

    

    for t in range(T):
        # returning the latent terms
        L_t = Lambdas[t] @ M_F
        latent_terms.append(L_t)

        Y_t = Thetas[t] @ Z_F + Lambdas[t] @ M_F
        Outcomes.append(Y_t)

    return Outcomes, latent_terms, M_F, Z_F



In the code above, the "latent terms" refer to the matrix product:
$$
\mathbf{\Lambda M}_F
$$
where $\mathbf{\Lambda}$ is the loading matrix for the latent factors and $\mathbf{M}_F$ are the latent factors for **all** the units (donors + treated). When we have no covariates, the latent terms are equal to the outcomes. If all goes as predicted, we will see 
$$
\mathbf{M}(\mathbf{Q}\vec{w}) = \vec{\mu}_1
$$
where $\mathbf{M}$ is the matrix of latent factors for the donors and $\vec{\mu}_1$ is the vector of latent factors for the treated unit.

# Set up for Single Time Point

For a single timepoint, the data is generated according to:
$$
\mathbf{Y} = \boldsymbol{\Theta}\mathbf{Z}_F + \boldsymbol{\Lambda}\mathbf{M}_F,
$$
and concretely, the characters involved can be visualized as:
$$
\underbrace{\begin{pmatrix}
y_{11} & y_{21} & y_{31}\\
y_{12} & y_{22} & y_{32}
\end{pmatrix}}_{\mathbf{Y}_F} =
\underbrace{\begin{pmatrix}
\theta_{11} &\theta_{21}\\
\theta_{12} &\theta_{22}
\end{pmatrix}}_{\Theta} 
\underbrace{\begin{pmatrix}
z_{11} & z_{21} & z_{31}\\
z_{12} & z_{22} & z_{32}
\end{pmatrix}}_{\mathbf{Z}_F} + 
\underbrace{\begin{pmatrix}
\lambda_{11} &\lambda_{21}\\
\lambda_{12} &\lambda_{22}
\end{pmatrix}}_{\Lambda} 
\underbrace{\begin{pmatrix}
\mu_{11} & \mu_{21} & \mu_{31}\\
\mu_{12} & \mu_{22} & \mu_{32}
\end{pmatrix}}_{\mathbf{M}_F}
$$


# Fitting on Data Generated without Covariates (Latent Factors only Case)

In [321]:
torch.manual_seed(42)

# settings
T = 1
N = 7
m = 2
f = 3
# since we aren't passing covariates, p doesn't matter here.
p = 2

with_covariates = False

# run the DGP without covariates - don't need Z_F
Outcomes, latent_terms, M_F,_ = sim_Data(T, N, m, p, f, with_covariates)
# these are the same cause there are no covariates

donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# prepare targets and covariates
targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=False)

print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)


M@gamma: 
 tensor([-1.4311, -1.0993,  0.4646], dtype=torch.float64)
mu_1:
 tensor([-1.2405, -0.9017,  0.5781], dtype=torch.float64)


The matrix $\mathbf{Q}$ is learning a representation of the donor outcomes $\mathbf{Y}$ that contain the target vector of outcomes $\vec{y}$ in their convex hull. This happens in such a way that $\mathbf{Q}$ transforms the latent donor factors to contain the target latent factors in their convex hull.

Why does this happen?

Let $\vec{\gamma}= \mathbf{Q}\vec{w}$. The objective is to minimize:
$$
\min_{\vec{w}}||\vec{y} - \mathbf{Y}(\mathbf{Q}\vec{w})|| = \min_{\vec{\gamma}}||\vec{y} - \mathbf{Y}\vec{\gamma}||,
$$ 
where $\mathbf{Y}$ represent the outcomes for the donors, and $\vec{y}$ are the outcomes for the treated unit. Since we are using only 1 timepoint, we supress the subscript $t$. By hypothesis (latent factors only, no covariates), we have the following:
$$
\vec{y} = \Lambda \vec{\mu_1}\quad\text{and}\quad \mathbf{Y} = \Lambda\mathbf{M},
$$ 
where $\vec{\mu}_1$ is the first column of the matrix $M_F$, which are the latent factors for the treated unit. Writing out the objective, we have:
$$
\min_{\vec{\gamma}}||\Lambda \vec{\mu}_1 - \Lambda M \vec{\gamma}|| = \min_{\vec{\gamma}}||\Lambda (\vec{\mu}_1 - M \vec{\gamma})|| = \min_{\vec{\gamma}} [\Lambda (\vec{\mu} - \mathbf{M}\vec{\gamma})]^T [\Lambda (\vec{\mu} - \mathbf{M}\vec{z})],
$$ 
which can be expressed as:
$$
\min_{\vec{\gamma}} (\vec{\mu}_1 - \mathbf{M}\vec{\gamma})^T(\Lambda^T\Lambda) (\vec{\mu}_1 - \mathbf{M}\vec{\gamma}).
$$
Since $\Lambda^T\Lambda$ is a constant, the minimum occurs when $\vec{\mu}_1 =\mathbf{M}\vec{\gamma}$. In otherwords, 
$$
\vec{\mu}_1 = (\mathbf{M}\mathbf{Q})\vec{w},
$$ 
meaning that $\mathbf{Q}$ learns a representation of the latent factors of the donor units which contains the treated units latent factors in its convex hull. 

The question now is how this generalizes. Now, we consider the case where the data is generated with covariates as well.

# Fitting on Data Generated with Covariates

### Passing the Covariates to the model

In [318]:
torch.manual_seed(42)

# settings
T = 1
N = 7
m = 2
f = 2
p = 2
# Generating data with covariates.
with_covariates = True

# we use Z_F now because we are passing covariates
Outcomes, latent_terms, M_F, Z_F = sim_Data(T, N, m, p, f, with_covariates)

# the latent factors, we will test to see if the method recovers these
donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# Combining the outcomes with the covariates, so they can be passed together
Outcomes_tensor = Outcomes[0].detach().clone()
#print("Outcomes:\n", Outcomes_tensor)
Z_F_tensor = Z_F.detach().clone()
#print("Z_F:\n", Z_F_tensor)
stacked = torch.cat([Outcomes_tensor, Z_F_tensor], dim = 0)
#print("Passed to model:\n", stacked)

# prepare targets and covariates
targets = []
covariates = []
for outcome in stacked.unsqueeze(0):
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

#print("Targets:\n", targets)
#print("Covariates:\n", covariates)

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=False)

# Hypothesis is that gamma times the donor latent factors will equal the latent factors for the control - THIS IS TRUE WHEN THE NUMBER OF DONORS IS LARGER THAN NUMBER OF COVARIATES?
# Gamma times donor latent factors:
print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)


M@gamma: 
 tensor([-0.0085, -0.2087], dtype=torch.float64)
mu_1:
 tensor([-0.0085, -0.2089], dtype=torch.float64)


The simulation above shows: with 1 time point, and data generated under latent factor model 
$$
\mathbf{Y}_t = \boldsymbol{\Theta}_t\mathbf{Z}_F + \boldsymbol{\Lambda}_t \mathbf{M}_F,
$$ 
the procedure learns a matrix $\mathbf{Q}$ that perfectly recovers the latent factors $\mathbf{M}$, even when these latent factors are combined with covariate info in the outcomes.

How does this work? When we are passing the covariates to the model, the objective can be expressed as:
\begin{equation}
\min_{\vec{\gamma}} \bigg|\bigg|\begin{pmatrix}
\vec{y}\\\vec{z}
\end{pmatrix} - 
\begin{pmatrix}
\mathbf{Y}\\ \mathbf{Z}
\end{pmatrix}\vec{\gamma} \bigg|\bigg|
\end{equation}
where $\vec{z}$ are the covariates for the treated unit, and $\mathbf{Z}$ are the covariates for the donors.
For a vector $\vec{\gamma}$ to be a solution to this minimization problem 
$$
\vec{z}-\mathbf{Z}\vec{\gamma} \approx 0

$$
must hold. This requirement is equivalent to:
$$
\mathbf{\Theta}\vec{z} - \mathbf{\Theta}\mathbf{Z}\vec{\gamma} \approx 0.
$$ 
The trick is to exchange the double minimization problem written above (equation 1) for the single equivalent **constrained** minimization:
$$
\min_{\vec{\gamma}} ||\vec{y} - \mathbf{Y}\vec{\gamma}|| \quad\text{subject to:} \quad\mathbf{\Theta}\vec{z} - \mathbf{\Theta}\mathbf{Z}\gamma \approx 0.
$$
And since the data is generated under $\mathbf{Y}_F = \mathbf{\Theta Z}_F + \mathbf{\Lambda M}_F$ we can write this as:
$$
\min_{\vec{\gamma}} || (\underbrace{\mathbf{\Theta}\vec{z} + \mathbf{\Lambda}\vec{\mu}_1}_{\vec{y}}) - \underbrace{(\mathbf{\Theta}\mathbf{Z} +\mathbf{\Lambda}\mathbf{M})}_{\mathbf{Y}}\vec{\gamma}|| = \min_{\vec{\gamma}} || (\underbrace{\mathbf{\Theta}\vec{z} + \mathbf{\Lambda}\vec{\mu}_1}_{\vec{y}}) - (\mathbf{\Theta}\mathbf{Z} +\mathbf{\Lambda}\mathbf{M})\vec{\gamma}|| = \min_{\vec{\gamma}} ||\underbrace{(\mathbf{\Theta}\vec{z} - \mathbf{\Theta}\mathbf{Z}\vec{\gamma})}_{0} +  (\mathbf{\Lambda}\vec{\mu}_1 - \mathbf{\Lambda}\mathbf{M}\vec{\gamma}) ||
$$ 


and the covariate (z) terms are killed off by the constraint, the problem becomes simply
$$
\min_{\vec{\gamma}}||\mathbf{\Lambda}\vec{\mu}_1 - \mathbf{\Lambda M}\vec{\gamma}|| = \min_{\vec{\gamma}} ||\mathbf{\Lambda}(\vec{\mu}_1 -\mathbf{M}\vec{\gamma})||
$$
which we showed above is a minimization which recovers the latent factors $\mathbf{M}$.

Does this still hold, given the same data, when we do not pass the covariates? This removes the constraint $\mathbf{\Theta}\vec{z} - \mathbf{\Theta}\mathbf{Z}\vec{\gamma} \approx 0,$ so we have no reason to expect the latent factors will be recovered.

### Not Passing Covariates to the Model

In [308]:
# USING THE SAME SETTINGS AS ABOVE, ONLY DIFFERENCE IS NOT PASSING COVARIATES.
torch.manual_seed(42)

# the latent factors, we will test to see if the method recovers these
donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

# Not passing covariates, just outcomes - data is generated with covariates, so this should be harder!
Outcomes_tensor = Outcomes[0].detach().clone()

# prepare targets and covariates
targets = []
covariates = []
for outcome in Outcomes_tensor.unsqueeze(0):
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=False)

# Gamma times donor latent factors:
print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)

M@gamma: 
 tensor([-0.5032, -1.4116], dtype=torch.float64)
mu_1:
 tensor([-0.3870, -0.2254], dtype=torch.float64)


We can see that indeed, the latent factors are **not** recovered. In fact, without passing the covariates, no matter how many donors we use, the latent factors are not recovered. This makes sense based on the derivation above. The basic idea is that the covraiates guide the algorithm to uncovering the latent pattern. Without the covariate information, the algorithm finds a solution that minimizes the loss, but in some sense it is not the "right" solution because it does not leverage the underlying data-generating information.

Now, how does this all work when we observe outcomes at multiple timepoints? 

# Multiple Timepoints

In [317]:
torch.manual_seed(42)

# settings
T = 20
N = 7
m = 2
f = 2
p = 2
# Generating data with covariates.
with_covariates = True

# we use Z_F now because we are passing covariates
# note, Z_F, M_F are constant across time. 
# we also do not care about the latent terms here.
Outcomes, latent_terms , M_F, Z_F = sim_Data(T, N, m, p, f, with_covariates)

# the latent factors, we will test to see if the method recovers these
donor_latent_factors = M_F[:,1:]
treated_latent_factors = M_F[:,0]

outcomes_tensor = torch.stack((Outcomes[0], Outcomes[1]), dim = 1)
#print("outcomes_tensor\n", outcomes_tensor)
# Combining the outcomes with the covariates, so they can be passed together
outcomes_detached = outcomes_tensor.detach().clone()
#print("outcomes_detached:\n", outcomes_detached)

Z_F_tensor = Z_F.detach().clone()
#print("Z_F:\n", Z_F_tensor)
Z_F_unsqueezed = Z_F_tensor.unsqueeze(0)
#print("Z_F_unsqueezed:\n", Z_F_unsqueezed)
#print("outcomes_detached.shape:", outcomes_detached.shape)
Z_F_expanded = Z_F_unsqueezed.expand(outcomes_detached.shape[0], -1, -1)  # [2, 2, 3]
#print("Z_F_expanded:\n", Z_F_expanded)

# BALANCE COVARIATES AT EVERY TIME POINT!
# so they've got to be combined with outcomes for each timepoint.
stacked = torch.cat([outcomes_detached, Z_F_expanded], dim=1)  # [2, 4, 3]
# print("stacked:\n", stacked)

# This is the correct shape, because there are two timepoints - so we will have two covies of what we had before.
# print("Passed to model:\n", stacked.squeeze(0))

# prepare targets and covariates
targets = []
covariates = []
for outcome in stacked.squeeze(0):
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

# print("Targets:\n", targets)
# print("Covariates:\n", covariates)

# run the learning algorithm
Q_final, w_final = lq.learnQ(targets, covariates, embedding_dim=5, n_iterations=2000, reg_Q=0, reg_w=1, verbose=False)

# Gamma times donor latent factors:
print("M@gamma: \n", donor_latent_factors @ (Q_final @ w_final))
print("mu_1:\n", treated_latent_factors)

M@gamma: 
 tensor([-0.2387, -0.0909], dtype=torch.float64)
mu_1:
 tensor([-0.2387, -0.0909], dtype=torch.float64)


The latent information is recovered in the multiple time points case.

This result actually makes a lot of sense. The main characters in the story are the outcomes at the two timepoints:
$$
\mathbf{Y}_{F1} = \mathbf{\Theta}_1 \mathbf{Z}_F + \mathbf{\Lambda}_1\mathbf{M}_F \quad\text{and}\quad \mathbf{Y}_{F2} = \mathbf{\Theta}_2 \mathbf{Z}_F + \mathbf{\Lambda}_2\mathbf{M}_F.
$$
When we pass covariates to the model, the objective is:
$$
\min_{\vec{\gamma}} ||\vec{y}_1 - \mathbf{Y}_1\vec{\gamma}|| + ||\vec{y}_2 - \mathbf{Y}_2\vec{\gamma}||\quad\text{subject to:}\quad \vec{z} - \mathbf{Z}_F \vec{\gamma}\approx 0.
$$
Because of the constraints on the covariate terms, as in the previous derivation the objective simplifies to:
$$
\min_{\vec{\gamma}} ||\mathbf{\Lambda}_1 (\vec{\mu}_1 - \mathbf{M}\vec{\gamma})|| + ||\mathbf{\Lambda}_2 (\vec{\mu}_1 - \mathbf{M}\vec{\gamma})||.
$$ 
But since the underlying latent factors ($\mathbf{M}$ and $\vec{\mu}_1$) do not change, finding $\vec{\gamma}$ that solves the minimization the problem is only as hard as finding a solution in the single timepoint case. 

We find experimentally that the overall loss is independent of the number of timepoints, which builds our confidence in the above calculation.

# Eigenvalues?

Given the two constraints
$$
\vec{z} - \mathbf{Z}_F \vec{\gamma}\approx 0 \quad\text{and}\quad\vec{\mu}_1 - \mathbf{M}\vec{\gamma} \approx 0
$$
what can we infer about $\mathbf{Q}$ using eigen-calculations?

$$
\vec{z} - (\mathbf{Z}_F\mathbf{Q})\vec{w} = 0\quad\text{and}\quad \vec{\mu}_1 - (\mathbf{M}\mathbf{Q})\vec{w}=0
$$

# Important Question

**A VERY important question is whether the brute force approach similarly recovers the hidden latent information.**